In [ ]:
# RSNA Knee 2.5D MEDICAL ENSEMBLE (Kaggle CPU, internet OFF): RadImageNet ResNet50 + DenseNet121,
# both on 2.5D adjacent-slice triplets, model-level weighted average. The 0.93-cluster recipe.
import os, glob, time, numpy as np, pandas as pd, torch, torch.nn as nn, torch.nn.functional as F
from torchvision.models import resnet50, densenet121
import pydicom, cv2
torch.set_num_threads(os.cpu_count() or 4)
K=40; IMG=384; RES=224; EMB_BS=48
W_RES, W_DEN = 0.6, 0.4   # ResNet50 is the proven 0.904 backbone; tune from gold-58
LABELS=['ACL','MCL','Medial Meniscus','Lateral Meniscus','Medial OA','Lateral OA','PF OA',
        'Effusion','Synovitis',"Baker's",'Contusion','Fracture']
dev='cuda' if torch.cuda.is_available() else 'cpu'
print('torch',torch.__version__,'| device',dev)
base=None
for r in sorted(glob.glob('/kaggle/input/*')):
    h=glob.glob(r+'/**/test_series.csv', recursive=True)
    if h: base=os.path.dirname(h[0]); break
print('base',base)
test_series=pd.read_csv(os.path.join(base,'test_series.csv'))
sub=pd.read_csv(os.path.join(base,'sample_submission.csv'))
print('test studies', sub.shape[0])

In [ ]:
class EncR(nn.Module):
    def __init__(s): super().__init__(); s.backbone=nn.Sequential(*list(resnet50(weights=None).children())[:-2])
    def forward(s,x): return s.backbone(x).mean(dim=(2,3))
class EncD(nn.Module):
    def __init__(s): super().__init__(); s.backbone=nn.Sequential(densenet121(weights=None).features)
    def forward(s,x): return F.relu(s.backbone(x),inplace=True).mean(dim=(2,3))
rw=glob.glob('/kaggle/input/**/ResNet50_radimagenet.pt', recursive=True)[0]
dw=glob.glob('/kaggle/input/**/DenseNet121.pt', recursive=True)[0]
encR=EncR(); encR.load_state_dict(torch.load(rw,map_location='cpu'),strict=True); encR=encR.eval().to(dev)
encD=EncD(); sdd=torch.load(dw,map_location='cpu',weights_only=False); sdd=sdd.state_dict() if hasattr(sdd,'state_dict') else sdd
encD.load_state_dict(sdd,strict=False); encD=encD.eval().to(dev)
print('encoders loaded')
class MIL(nn.Module):
    def __init__(s,di,d=256,n=12):
        super().__init__(); s.proj=nn.Sequential(nn.Linear(di,d),nn.LayerNorm(d),nn.GELU())
        s.plane=nn.Embedding(3,d); s.fluid=nn.Embedding(2,d)
        s.V=nn.Linear(d,128); s.U=nn.Linear(d,128); s.w=nn.Linear(128,1)
        s.head=nn.Sequential(nn.Dropout(0.3), nn.Linear(d,n))
    def forward(s,x,pid,fb):
        h=s.proj(x)+s.plane(pid)+s.fluid(fb)
        a=torch.softmax(s.w(torch.tanh(s.V(h))*torch.sigmoid(s.U(h))),0)
        return s.head((a*h).sum(0))
def load_heads(pat):
    p=glob.glob('/kaggle/input/**/'+pat, recursive=True)[0]; ck=torch.load(p,map_location='cpu'); di=int(ck.get('di'))
    hs=[]
    for sd in ck['heads']:
        m=MIL(di=di).to(dev).eval(); m.load_state_dict(sd); hs.append(m)
    print(pat,'heads',len(hs),'di',di,'gold58',ck.get('gold58_ensemble')); return hs
headsR=load_heads('head_weights_radimagenet_25d.pt')
headsD=load_heads('head_weights_densenet121_25d.pt')
PL={'Sagittal':0,'Coronal':1,'Axial':2}
print('ready')

In [ ]:
def win(a):
    lo,hi=np.percentile(a,0.5),np.percentile(a,99.5); a=np.clip((a-lo)/max(hi-lo,1e-6),0,1)
    return cv2.resize(a,(IMG,IMG),interpolation=cv2.INTER_AREA).astype(np.float32)
def rd(p):
    d=pydicom.dcmread(p); a=d.pixel_array.astype(np.float32)
    return a*float(getattr(d,'RescaleSlope',1) or 1)+float(getattr(d,'RescaleIntercept',0) or 0)
def positions(n,k):
    if n<=0: return []
    if n<=k: return list(range(n))
    lo,hi=int(n*.12),int(n*.88)
    if hi<=lo: lo,hi=0,n
    return [int(round(x)) for x in np.linspace(lo,hi-1,k)]
def slc25(fs,idx):
    n=len(fs); out=[]
    for j in (idx-1,idx,idx+1):
        j=max(0,min(n-1,j))
        try: out.append(win(rd(fs[j])))
        except Exception: out.append(np.zeros((IMG,IMG),np.float32))
    return np.stack(out)
@torch.no_grad()
def emb(encf,triples):
    x=torch.from_numpy(np.stack(triples)); x=x*2.0-1.0
    if x.shape[-1]!=RES: x=F.interpolate(x,size=(RES,RES),mode='bilinear',align_corners=False)
    return encf(x.to(dev)).float()
@torch.no_grad()
def predict_study(study):
    trips=[];pids=[];fbs=[]
    for _,sr in test_series[test_series['StudyInstanceUID']==study].iterrows():
        sd=os.path.join(base,'test_series',study,sr['SeriesInstanceUID'])
        if not os.path.isdir(sd):
            alt=glob.glob(os.path.join(base,'**',sr['SeriesInstanceUID']),recursive=True); sd=alt[0] if alt else None
        if not sd: continue
        fs=sorted(glob.glob(sd+'/*.dcm')); pid=PL.get(sr['Anatomical_Plane'],0); fb=int(sr['Fluid_Sensitive'])
        for idx in positions(len(fs),K): trips.append(slc25(fs,idx)); pids.append(pid); fbs.append(fb)
    if not trips: return np.full(12,0.5)
    pid=torch.tensor(pids,device=dev); fb=torch.tensor(fbs,device=dev)
    XR=[];XD=[]
    for i in range(0,len(trips),EMB_BS):
        b=trips[i:i+EMB_BS]; XR.append(emb(encR,b)); XD.append(emb(encD,b))
    XR=torch.cat(XR,0); XD=torch.cat(XD,0)
    pR=np.mean([torch.sigmoid(h(XR,pid,fb)).cpu().numpy() for h in headsR],0)
    pD=np.mean([torch.sigmoid(h(XD,pid,fb)).cpu().numpy() for h in headsD],0)
    return W_RES*pR + W_DEN*pD
print('fns ready')

In [ ]:
ids=list(sub['StudyInstanceUID']); t0=time.time(); out={}
for i,s in enumerate(ids):
    out[s]=predict_study(s)
    if i%100==0: print(f'{i}/{len(ids)} {time.time()-t0:.0f}s',flush=True)
res=pd.DataFrame({'StudyInstanceUID':ids})
for j,l in enumerate(LABELS): res[l]=[out[s][j] for s in ids]
res=res[['StudyInstanceUID']+LABELS]; res.to_csv('submission.csv',index=False)
print('WROTE submission.csv',res.shape,f'{time.time()-t0:.0f}s'); print(res.head())